In [1]:
import torch
import torch.nn as nn
import os

In [2]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [3]:
class GramMatrix(nn.Module):
    def forward(self, x):
        b, c, h, w = x.size()
        features = x.view(b * c, h * w)
        G = torch.mm(features, features.t())
        return G.div(b * c * h * w)

In [4]:
class TotalVariationLoss(nn.Module):
    """Smooth noise tanpa menghapus edge."""
    def forward(self, x):
        h_var = torch.mean(torch.abs(x[:, :, 1:, :] - x[:, :, :-1, :]))
        w_var = torch.mean(torch.abs(x[:, :, :, 1:] - x[:, :, :, :-1]))
        return h_var + w_var
 


In [ ]:
class SobelEdgeLoss(nn.Module):
    
    def __init__(self):
        super().__init__()
        kx = torch.tensor([[-1, 0, 1], [-2, 0, 2], [-1, 0, 1]],
                           dtype=torch.float32).view(1, 1, 3, 3)
        ky = torch.tensor([[-1, -2, -1], [0, 0, 0], [1, 2, 1]],
                           dtype=torch.float32).view(1, 1, 3, 3)
        self.register_buffer('kx', kx)
        self.register_buffer('ky', ky)
 
    def forward(self, gen, target):
        ex = F.conv2d(gen,    self.kx, padding=1)
        ey = F.conv2d(gen,    self.ky, padding=1)
        tx = F.conv2d(target, self.kx, padding=1)
        ty = F.conv2d(target, self.ky, padding=1)
        mag_gen = torch.sqrt(ex**2 + ey**2 + 1e-8)
        mag_tar = torch.sqrt(tx**2 + ty**2 + 1e-8)
        return F.mse_loss(mag_gen, mag_tar)


In [ ]:
class GeospatialStyleTransfer(nn.Module):
    """
    VGG19-based feature extractor untuk grayscale heightmap.
    Style  : conv1_1, conv2_1, conv3_1, conv4_1  (multi-scale texture)
    Content: conv4_2                              (mid-level structure)
    """
    def __init__(self):
        super().__init__()
        # Block 1
        self.conv1_1 = nn.Conv2d(1, 64, 3, padding=1)
        self.conv1_2 = nn.Conv2d(64, 64, 3, padding=1)
        self.pool1   = nn.AvgPool2d(2, 2)
        # Block 2
        self.conv2_1 = nn.Conv2d(64, 128, 3, padding=1)
        self.conv2_2 = nn.Conv2d(128, 128, 3, padding=1)
        self.pool2   = nn.AvgPool2d(2, 2)
        # Block 3
        self.conv3_1 = nn.Conv2d(128, 256, 3, padding=1)
        self.conv3_2 = nn.Conv2d(256, 256, 3, padding=1)
        self.conv3_3 = nn.Conv2d(256, 256, 3, padding=1)
        self.conv3_4 = nn.Conv2d(256, 256, 3, padding=1)
        self.pool3   = nn.AvgPool2d(2, 2)
        # Block 4
        self.conv4_1 = nn.Conv2d(256, 512, 3, padding=1)
        self.conv4_2 = nn.Conv2d(512, 512, 3, padding=1)
        self.conv4_3 = nn.Conv2d(512, 512, 3, padding=1)
 
    def forward(self, x):
        r = {}
        r['conv1_1'] = F.relu(self.conv1_1(x))
        x = self.pool1(F.relu(self.conv1_2(r['conv1_1'])))
 
        r['conv2_1'] = F.relu(self.conv2_1(x))
        x = self.pool2(F.relu(self.conv2_2(r['conv2_1'])))
 
        r['conv3_1'] = F.relu(self.conv3_1(x))
        x = F.relu(self.conv3_2(r['conv3_1']))
        x = F.relu(self.conv3_3(x))
        x = self.pool3(F.relu(self.conv3_4(x)))
 
        r['conv4_1'] = F.relu(self.conv4_1(x))
        r['conv4_2'] = F.relu(self.conv4_2(r['conv4_1']))
        r['conv4_3'] = F.relu(self.conv4_3(r['conv4_2']))
 
        content_maps = [r['conv4_2']]
        style_maps   = [r['conv1_1'], r['conv2_1'], r['conv3_1'], r['conv4_1']]
        return content_maps, style_maps


In [ ]:
def load_and_inject_weights(model):
    weight_url  = 'https://download.pytorch.org/models/vgg19-dcbb9e9d.pth'
    weight_path = 'vgg19_raw.pth'
 
    if not os.path.exists(weight_path):
        print("Downloading VGG19 weights...")
        urllib.request.urlretrieve(weight_url, weight_path)
 
    raw = torch.load(weight_path, map_location='cpu')
    md  = model.state_dict()
 
    mapping = {
        'conv1_1': 'features.0',  'conv1_2': 'features.2',
        'conv2_1': 'features.5',  'conv2_2': 'features.7',
        'conv3_1': 'features.10', 'conv3_2': 'features.12',
        'conv3_3': 'features.14', 'conv3_4': 'features.16',
        'conv4_1': 'features.19', 'conv4_2': 'features.21',
        'conv4_3': 'features.23',
    }
 
    for custom, vgg in mapping.items():
        w = raw[f"{vgg}.weight"]
        b = raw[f"{vgg}.bias"]
        if custom == 'conv1_1':
            w = w.mean(dim=1, keepdim=True)
        md[f"{custom}.weight"].copy_(w)
        md[f"{custom}.bias"].copy_(b)
 
    for p in model.parameters():
        p.requires_grad_(False)
 
    print("Weight injection successful.")
    return model
 
GRAY_MEAN = 0.5
GRAY_STD  = 0.5
 
def normalize(x):
    return (x - GRAY_MEAN) / GRAY_STD

L-BFGS

In [ ]:
import torch.optim as optim
import torch.nn.functional as F
from torchvision import transforms
from torchvision.utils import save_image
from PIL import Image

In [ ]:
def calculation_mse_loss(generated, target):
    return torch.mean((generated - target) ** 2)

In [ ]:
def gradient_loss(gen, target):
     # horizontal gradient
    gen_dx = gen[:, :, :, 1:] - gen[:, :, :, :-1]
    tar_dx = target[:, :, :, 1:] - target[:, :, :, :-1]

    # vertical gradient
    gen_dy = gen[:, :, 1:, :] - gen[:, :, :-1, :]
    tar_dy = target[:, :, 1:, :] - target[:, :, :-1, :]

    loss_x = torch.mean((gen_dx - tar_dx) ** 2)
    loss_y = torch.mean((gen_dy - tar_dy) ** 2)

    return loss_x + loss_y

In [ ]:
def train_hybrid_heightmap(content_path, style_path,output_path):
    transform = transforms.Compose([
        transforms.Grayscale(num_output_channels=1),
        transforms.Resize((512, 512)),
        transforms.ToTensor()
    ])
    def load_img(img_path):
        img = Image.open(img_path)
        return transform(img).unsqueeze(0).to(device)
   
    print("Load data...")
    content_img = load_img(content_path)
    style_img = load_img(style_path)
    
    generated_img = nn.Parameter(
        torch.randn_like(content_img) * 0.1
    )
    
    model = GeospatialStyleTransfer().to(device).eval()
    model = load_and_inject_weights(model)
    
    gram_calc = GramMatrix().to(device)
    tv_loss_calc = TotalVariationLoss().to(device)
    
    print("Extracting features from image...")
    with torch.no_grad():
        norm_content = (content_img - 0.449) / 0.226
        norm_style = (style_img - 0.449) / 0.226
        
        target_content_features, _ = model(norm_content)
        _, target_style_features = model(norm_style)
        
        target_gram_matrices = [gram_calc(sf) for sf in target_style_features]
        
    optimizer = optim.LBFGS([generated_img], max_iter = 500, lr=1.0)
    
    alpha = 1
    beta = 1e9
    gamma = 5e-6
    delta = 10
    epsilon = 5
    
    print("Starting the L-BFGS Optimization Process...")
    run= [0]
    
    def laplacian(x):

        kernel = torch.tensor([
            [0, 1, 0],
            [1,-4, 1],
            [0, 1, 0]
        ], dtype=torch.float32, device=x.device)

        kernel = kernel.view(1,1,3,3)

        return F.conv2d(x, kernel, padding=1)

    def laplacian_loss(gen, target):

        gen_lap = laplacian(gen)
        tar_lap = laplacian(target)

        return torch.mean((gen_lap - tar_lap) ** 2)
    
    def closure():
        optimizer.zero_grad()
        generated_img.data.clamp_(0,1)
        norm_gen = (generated_img - 0.449) / 0.226
        gen_content_features, gen_style_features = model(norm_gen)
        content_loss = calculation_mse_loss(gen_content_features[0], target_content_features[0])
        
        style_loss = 0
        for gen_sf, target_gm in zip(gen_style_features, target_gram_matrices):
            gen_gm = gram_calc(gen_sf)
            style_loss += calculation_mse_loss(gen_gm, target_gm)
        
        tv_loss = tv_loss_calc(generated_img)
        
        morph_loss = gradient_loss(generated_img, content_img)
        lap_loss = laplacian_loss(generated_img, content_img)
        
        total_loss = (
            alpha * content_loss
            + beta * style_loss
            + gamma * tv_loss
            + delta * morph_loss
            + epsilon * lap_loss
        )
        total_loss.backward()
        
        run[0] += 1
        if run[0] % 50 == 0:
            print(f"Iteration: {run[0]:03d}/500 | Total Loss: {total_loss.item():.4f}")
        return total_loss
        
    optimizer.step(closure)
    
    generated_img.data.clamp_(0,1)
    save_image(generated_img, output_path)
    print(f"\nDone! Hybrid Heightmap has been successfully saved in: {output_path}")
        
        

In [ ]:
if __name__ == "__main__":
    CONTENT = "img\contents\PerlinMap_1.png" 
    STYLE = "img/style/Style_1.png"
    OUTPUT = "img/output/hybrid_terrain.png"
    
    train_hybrid_heightmap(CONTENT, STYLE, OUTPUT)

Loading images...
Weight injection successful.
Extracting target features...
Starting optimization (500 effective iterations)...
Iteration: 0020/500 | Loss: 0.0718
Iteration: 0100/500 | Loss: 0.0597
Iteration: 0200/500 | Loss: 0.0596
Iteration: 0300/500 | Loss: 0.0595
Iteration: 0400/500 | Loss: 0.0595
Iteration: 0500/500 | Loss: 0.0594

✓ Saved (histogram-matched): img/output/hybrid_terrain_v2.png
